# 02 — Conv2D (Convolutional) Network on MNIST

This notebook trains a **Convolutional Neural Network (CNN)** using `Conv2D` layers on the MNIST dataset.

CNNs exploit the **spatial structure** of images by learning local filters — much more efficient and accurate than Dense layers for image tasks.

**Architecture:**
```
Input (28×28×1)
  → Conv2D(32, 3×3) → MaxPooling
  → Conv2D(64, 3×3) → MaxPooling
  → Flatten → Dense(64) → Dense(10, softmax)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

## 1. Load & Preprocess Data

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalise and add channel dimension (required by Conv2D)
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0
x_train = x_train[..., np.newaxis]  # (60000, 28, 28, 1)
x_test  = x_test[...,  np.newaxis]  # (10000, 28, 28, 1)

print(f'Train: {x_train.shape}, Test: {x_test.shape}')

## 2. Build the CNN Model

In [ ]:
def build_cnn_model():
    model = keras.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Classifier head
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ], name='CNN_Conv2D')
    return model

model = build_cnn_model()
model.summary()

## 3. Compile & Train

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start = time.time()
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)
elapsed = time.time() - start
print(f'\nTotal training time: {elapsed:.1f}s')

## 4. Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')
print(f'Total Params  : {model.count_params():,}')
print(f'Training Time : {elapsed:.1f}s')

## 5. Visualise Learned Filters

Let's peek at what the first Conv2D layer has learned.

In [ ]:
filters, _ = model.layers[0].get_weights()
# filters shape: (3, 3, 1, 32)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[:, :, 0, i], cmap='viridis')
    ax.axis('off')
plt.suptitle('Learned Conv2D Filters (Layer 1)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('CNN — Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('CNN — Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
import json, os

os.makedirs('../results', exist_ok=True)
results = {
    'model': 'CNN (Conv2D)',
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'total_params': model.count_params(),
    'training_time_s': round(elapsed, 1),
    'history': {
        'accuracy':     history.history['accuracy'],
        'val_accuracy': history.history['val_accuracy'],
        'loss':         history.history['loss'],
        'val_loss':     history.history['val_loss'],
    }
}

with open('../results/conv2d_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to results/conv2d_results.json')